In [1]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
import ast
import os

# ==========================================
# 1. CONFIGURARE
# ==========================================
INPUT_FILE = '13k-recipes.csv'
OUTPUT_FILE = 'recipes_translated_fast.csv'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 16 # Procesăm câte 16 texte deodată pentru viteză

print(f"Folosim dispozitivul: {DEVICE}") # Ar trebui să scrie 'cuda' pe serverul tău

# ==========================================
# 2. ÎNCĂRCARE MODEL LOCAL (Helsinki-NLP)
# ==========================================
model_name = 'Helsinki-NLP/opus-mt-en-ro'
print("Se încarcă modelul de traducere... (poate dura 1 minut)")
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(DEVICE)

# ==========================================
# 3. FUNCȚII DE TRADUCERE
# ==========================================

def translate_batch(texts):
    """Traduce o listă de texte simultan folosind GPU/CPU."""
    # Filtrăm textele goale sau non-string
    valid_texts = [str(t) if t else "" for t in texts]
    
    # Dacă lista e goală, returnăm gol
    if not valid_texts:
        return []

    # Tokenizare
    inputs = tokenizer(valid_texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
    
    # Generare traducere
    with torch.no_grad(): # Nu avem nevoie de gradienți (nu antrenăm, doar folosim)
        translated = model.generate(**inputs)
    
    # Decodare înapoi în text
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

def clean_text(text):
    if not isinstance(text, str): return ""
    return text.replace("tsp.", "teaspoon").replace("tbsp.", "tablespoon").replace("lb.", "pound")

# ==========================================
# 4. PROCESARE
# ==========================================

# Încărcare date
df = pd.read_csv(INPUT_FILE)
# df = df.head(50) # DECOMENTEAZĂ AICI PENTRU UN TEST RAPID CU 50 REȚETE

# Verificare checkpoint
start_index = 0
if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_csv(OUTPUT_FILE)
    start_index = len(df_existing)
    print(f"Reluăm de la linia {start_index}...")
else:
    pd.DataFrame(columns=df.columns).to_csv(OUTPUT_FILE, index=False)

# Buffer pentru scriere
temp_rows = []
rows_buffer = [] 

print("Începe traducerea rapidă...")

# Iterăm prin dataframe
for index, row in tqdm(df.iloc[start_index:].iterrows(), total=len(df)-start_index):
    
    # Curățăm puțin textul
    row['Title'] = clean_text(row['Title'])
    row['Instructions'] = clean_text(row['Instructions'])
    
    # Parsăm ingredientele
    try:
        ingredients_list = ast.literal_eval(row['Ingredients'])
        # Le unim temporar cu un separator unic pentru a le traduce într-un singur batch
        # Traducerea listelor element cu element e lentă, așa că le unim: "salt || pepper || oil"
        ingredients_str = " || ".join([clean_text(i) for i in ingredients_list])
    except:
        ingredients_str = clean_text(str(row['Ingredients']))
        ingredients_list = []

    # Adăugăm rândul în buffer pentru procesare
    # Structura în buffer va fi: [Titlu, Instrucțiuni, Ingrediente_String]
    rows_buffer.append({
        'original_row': row, 
        'texts_to_translate': [row['Title'], row['Instructions'], ingredients_str]
    })

    # Când bufferul atinge BATCH_SIZE, trimitem la GPU
    if len(rows_buffer) >= BATCH_SIZE:
        # Aplatizăm lista pentru traducător: [T1, I1, Ing1, T2, I2, Ing2, ...]
        flat_texts = []
        for item in rows_buffer:
            flat_texts.extend(item['texts_to_translate'])
        
        # Traducem tot calupul o dată!
        translated_flat = translate_batch(flat_texts)
        
        # Reconstruim rândurile
        for i, item in enumerate(rows_buffer):
            row = item['original_row']
            
            # Extragem traducerile (sunt grupate câte 3)
            base_idx = i * 3
            row['Title'] = translated_flat[base_idx]
            row['Instructions'] = translated_flat[base_idx + 1]
            
            # Refacem lista de ingrediente din string-ul unit
            ing_trans_str = translated_flat[base_idx + 2]
            row['Ingredients'] = str(ing_trans_str.split(" || "))
            
            # Gestionăm și Cleaned_Ingredients dacă există
            if 'Cleaned_Ingredients' in row:
                # Copiem traducerea de la Ingredients pentru simplitate, sau o lăsăm netradusă
                # De obicei Cleaned_Ingredients e folosită doar pentru analiză tehnică
                pass 

            temp_rows.append(row)

        # Salvare în CSV
        pd.DataFrame(temp_rows).to_csv(OUTPUT_FILE, mode='a', header=False, index=False)
        temp_rows = []
        rows_buffer = []

# Procesăm ce a rămas în buffer la final
if rows_buffer:
    flat_texts = []
    for item in rows_buffer:
        flat_texts.extend(item['texts_to_translate'])
    
    translated_flat = translate_batch(flat_texts)
    
    for i, item in enumerate(rows_buffer):
        row = item['original_row']
        base_idx = i * 3
        row['Title'] = translated_flat[base_idx]
        row['Instructions'] = translated_flat[base_idx + 1]
        ing_trans_str = translated_flat[base_idx + 2]
        row['Ingredients'] = str(ing_trans_str.split(" || "))
        temp_rows.append(row)
        
    pd.DataFrame(temp_rows).to_csv(OUTPUT_FILE, mode='a', header=False, index=False)

print(f"Gata! Fișier salvat: {OUTPUT_FILE}")

/bigdata/userhome/students/md5_sd7aa1b0bd2cf032768f024455ba/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Folosim dispozitivul: cuda
Se încarcă modelul de traducere... (poate dura 1 minut)


/bigdata/userhome/students/md5_sd7aa1b0bd2cf032768f024455ba/venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Începe traducerea rapidă...


100%|██████████| 13501/13501 [1:24:05<00:00,  2.68it/s]


Gata! Fișier salvat: recipes_translated_fast.csv
